GPU COMPUTING CON CUDA

GPU computing con CUDA è uno dei motivi per cui il deep learning moderno esiste.
Senza GPU:
    - addestrare reti grandi sarebbe lentissimo
    - gli LLM moderni sarebbero quasi impraticabili

La CPU è progettata per:
- logica complessa
- operazioni sequenziali
- pochi trread molto potenti
Ha pochi core, molto sofisticati
La GPU è progettata per:
- operazioni massive parallele
- stesso calcolo ripetuto tantissime volte
Ha migliaia di core piccoli
Ed il Deep Learning è praticamente moltiplicazioni gigantesche di matrici

Un pezzo di hardware nato per i videogiochi è diventato fondamentale dell'ai

Le GPU sono perfette per l'ai perchè eseguono migliaia di operazioni in parallelo.

CUDA significa Comput Unified Device Architecture
E' la piattaforma di NVIDIA che permette di usare la GPU per calcolo generale, non solo grafica.
Prima le GPU servivano quasi solo per video giochi o rendering, CUDA le ha trasformate in acceleratori matematici general purpose.
Con CUDA puoi scrivere codice che gira direttamente sulla GPU
Nel Deep Learning lo fanno framework come:
- Pytorch
- TensorFlow
Scrivi:
tensor.cuda()
oppure
tensor.to("cuda")
e il tensore viene spostato dalla RAM alla VRAM GPU


- Architettura parallela
- Memoria e trasferimetno
- Ecosistema software

Le GPU, nate per i videogiochi (reindirizzare immagini), sono architetture con migliaia di piccoli CORE. Mentre la CPU è un solita, la GPU è un'orchestra immensa.

CPU vs GPU
Gestione complessa contro forza bruta
- Core CPU: pochi ma estremamente potenti, ottimizzati per la logica di controllo (if-else) e la bassa latenza di esecuzione singola. Incridibilmente intelligente.
- Core GPU: migliaia di unità di calcolo semplificate, progettate per massimizzare il throughput totale piuttosto che la velocità del singolo task. Come studneti di scuole elementari che fanno solo operazioni semplici. Usa quasi tutto il silicio per le ALU unità aritmetriche
- Architettura Many-Core: permette di dividere una moltiplicazione tra matrici in migliaia di mini-operazioni indipendenti eseguite in parallelo.
- ALU Massive: la maggior parte dell'area del silicio in una GPU è dedicata al calcolo matematico, riducendo lo spazio per cache e predizione di salti.

Come coordiniamo questo esercito di piccoli calcolatori (GPU)
Paradigma di SIMT
- Single Istruction, Multiple Threads (SIMT): CUDA utilizza il modello SIMT, dove lo stesso comando viene impartito a un gruppo di thread che lo eseguono su dati diversi. E' l'essenza del calcolo tensoriale moderno
- Warp e Streaming Multiprocessors: I core sono raggruppati in SM (streaming multiprocessors). Un gruppo di 32 thread, chiamato Warp, esegue istruzioni in modo perfettamente sincrono.
- Vantaggio per le ANN: Poichè ogni neurone in un layer esegue la stessa operazione matematica, la GPU può calcolare un intero layer in un unico ciclo di clock collettivo

Efficienza Computazionale
Calcolo del rendimento parallelo
La capacità di una GPU di accelerare un algoritmo dipende da quanto il problema è parallelizzabile. Le reti neurali sono considerate problemi imbarazzantemente paralleli.
Il numero di operazioni in virgola mobile al secondo (FLOPS) di una GPU moderna supera di ordini di grandezza quello di una CPU per compiti tensoriali.
Però tutta questa potenza è inutile se non riusciamo a far arrivare i dati alla GPU in tempo

Trasferimento Host-Device
Il collo di bottiglia del bus PCle
In un sistema CUDA chiamiamo Host la CPU e Device la GPU. Il dato deve viaggiare dalla memoria RAM di sistema alla memoria VRAM della scheda video per essere elaborato.
Possiamo avere la GPU più potente del mondo, ma se la strada tra CPU e GPU è stretta e trafficata la nostra GPU passerà la metà del tempo ad aspettare che arrivano i dati.
Questo trasferimento avviene tramite il bus PCI Express e rappresenta spesso il punto più lento dell'intera pipeline di addestramento.

Gestione della memoria Video
VRAM e Gerarchie
La momeria video (VRAM) è dove risiede tutto il nostro modello, dobbiamo allocare spazio, caricare i pesi (dall'host ai device) ed alla fine riportare i risultati alla CPU (dal device all'host). Meno viaggi facciamo meglio è. Ecco perchè carichiamo i dati a blocchi (batch)
- Allocazione: prima di ogni calcolo dobbiamo riservare spazio nella VRAM del Device per ospitare i pesi del modello e i batch di dati.
- H2D (Hosto to Device): l'operazione di caricamente dei dati. E' fondamentale minimizzare la frequenza di questi passaggi per non penalizzare le performance.
- D2H (Device to Host): l'operazione di recupero dei risultati (es. Loss o le predizioni) per essere mostrati o salvati dalla CPU
- Memoria unificata: tecnologia moderna che permette a CPU e GPU di vedere lo stesso spazio di indirizzamento, semplificando la programmazione.

Come possiamo rendere questo traffico più fluido
Ottimizzazione del Flusso
- Data Pinning: l'uso di memoria 'pinned' (non paginabile) velocizza il trasferimento H2d poichè permette al controller DMA di copiare i dati senza l'intervento della CPU. Prepariamo i dati in RAM in modo che siano pronti in una corsia preferenziale verso la GPU
- Asincronia dei Kernel: CUDA permette di inviare comandi alla GPU e continuare l'esecuzione sulla CPU. Questo consente di sovrapporre il calcolo di un batch con il caricamento del successivo. Mentre la GPU sta calcolando batch n.1 la CPU sta caricando il batch n.2
- Bandwidth vs Latenza: la larghezza di banda determina quanti dati passano, ma la latenza determina quanto tempo passa prima che il primo byte arrivi. Nel Deep Learning puntiamo alla banda larga.

Tempo totale di esecuzione
Il costo del movimento dati
Se il tempo di trasferimento è superiore al tempo di calcolo, la GPU resterà inattiva (idle) aspettando i dati. Questo è il motivo per cui usiamo batch di grandi dimensioni.
T(totale)=T(trasferimento)+T(compute)+T(overlap)
somma di trasferimenti, attesi, calcoli
Un batch più grande permette alla GPU di lavorare più a lungo su un unico trasferimento, ammortizando il costo del viaggio sul bus PCle. Nascondere il tempo di trasferimento dietro il tempo di calcolo.


Stack Software: CUDE e cuDNN
Tradurre la matematica in silicio
Programmare direttamente i core di una GPU sarebbe estremamente complesso. Per questo utilizziamo CUDA come linguaggio di interfaccia e cuDNN come libreria di algoritmi ottimizzati.
Questi strumenti si pongono tra i framework come PyTorch/TensorFlow e l'hardware, garantendo che le operazioni siano eseguite nel modo più efficiente possibile.

CUDA Toolkit
L'ecosistema di sviluppo CUDA
- Driver CUDA: il software di base che permette al sistema operativo di comunicare con il chip grafico e gestire le risorse
- Runtime API: le funzioni utilizzate dai framework per lanciare i calcoli (kernel) sulla GPU in modo trasparente per l'utente
- NVCC: il compilatore che trasforma il codice C++/CUDA in istruzioni binarie comprensibili dalla scheda video.
- Parallelismo di massa: CUDA gestisce la distribuzione automatica dei thread sui core disponibili, garantendo la scalabilità tra diverse schede.

La magia di cuDNN
cuDNN è il vero segreto di pulcinella del Deep Learning.
Questa libreria non solo esegue la matematica ma fa autotuning, testa diversi modi di fare una convoluzione e sceglie la migliore per la scheda video a disposizione. Ottimizza ogni singolo layer della rete neurale, mentra stiamo addestrando.
- Deep Neural Network Library: cuDNN fornisce implementazioni altamente ottimizzate per operazioni standard come convoluzioni, pooling e funzioni di attivazione.
- Autotuning: la libreria è in grado di testare diversi algoritmi per una stessa operazione (es. convoluzione FFT vs Winograd) e scegliere il più veloce per quell'hardware specifico.
- Astrazione Totale: grazie a cuDNN, non dobbiamo preoccuparci della geometria dei registri o della memoria condivisa; il framework gestisce queste ottimizzazioni per noi

Precisione e Tensor Cores
Accelerazione hardware specifica
Le GPU moderne includono i Tensor Cores, unità specializzate solo nelle moltiplicazioni di matrici 4x4 in un unico ciclo (in un colpo solo). cuDNN sfrutta queste unità per accelerare drasticamente il training.
L'uso della mezza precisione (Float16), riducendo la precisione dei decimali, permette di raddopiare la velcoità di calcolo e dimezzare l'occupazione di memoria senza perdere accuratezza significativa.

concludendo il punto debole è il trasferiemnto dei dati e deve essere ottimizzato con batch (grandi) di adeguate dimensioni e pipeline sincrone

La GPU accelera i calcoli, non rende il modello più intelligente

La GPU però non è obbligatoria. Se fai studio, rete reletivamente piccole, dataset medi la CPU basta. La GPU diventa davvero importante con CNN grandi, Transformer, LLM, dataset enormi.
Per Pytorch CUDA Windows funziona bene
Per TensorFlow supporto GPU Windows nativo quasi morto, meglio WSL2 Linux

Per Deep Learning serio NVIDIA è praticamente obbligatoria, AMD nel deep learning supportata molto peggiore